# Spark Lakehouse — Retail Data Unification
## Gold Layer: Dimensional Modelling and Analytics-Ready Data

### 1. Business Context

The Bronze and Silver layers ingest, standardize, and unify retail data from online commerce and offline Point-of-Sale (POS) systems. The Gold layer transforms these unified datasets into structures suitable for reporting and analytical workloads.

### 2. Objectives

- Maintain a consolidated customer dimension using SCD Type 1.
- Maintain product history using Slowly Changing Dimension Type 2 (SCD Type 2).
- Build a fact table containing orders enriched with customer and product surrogate keys.
- Select the product version valid for the order's processing date.
- Identify and quarantine orders with missing customer or product lookup records.
- Process data using a configurable file-run date.

### 3. Gold Layer Components

| Component | Purpose |
|---|---|
| `gold.customers` | Customer dimension containing unified customer attributes and a customer surrogate key |
| `gold.products` | Product dimension preserving historical versions using effective dates and a current-record flag |
| `gold.fact_orders` | Order fact table enriched with customer and product surrogate keys |
| `silver.reject_orders` | Reject table for orders whose customer or product lookup fails |

### 4. Dimensional Modelling Approach

**Customer dimension — SCD Type 1:** Updates existing customer attributes rather than maintaining previous attribute versions. The customer surrogate key is derived from the mobile number using SHA-256 hashing.

**Product dimension — SCD Type 2:** Preserves product history by expiring the previous current record when product attributes change and inserting a new version. The product surrogate key is derived from the SKU and file date.

**Order fact table:** Associates each order with the relevant customer and product dimension records using their surrogate keys.

### 5. Technology Stack

**Databricks · Spark SQL · Delta Lake · MERGE · Window Functions · SHA-256 Hashing**

### 6. Expected Outcome

A dimensional data model that combines customer, product, and order information into analytics-ready Delta tables while retaining product history and isolating orders with unresolved dimension lookups.

In [0]:

from datetime import date, timedelta
from pyspark.sql.functions import *

yesterday = date.today() - timedelta(days=1)
yesterday_str = yesterday.strftime("%Y%m%d")

dbutils.widgets.text("file_run_date", yesterday_str)

file_run_date = dbutils.widgets.get("file_run_date")
print(file_run_date)


# dbutils.widgets.text("file_run_date", "20260207")
# file_run_date = dbutils.widgets.get("file_run_date")

20260207


In [0]:
%skip
delete from `spark-databricks-project-catalog`.gold.customers;
delete from `spark-databricks-project-catalog`.gold.products;
delete from `spark-databricks-project-catalog`.gold.fact_orders;

num_affected_rows
1704233


## 1. Customer Dimension — SCD Type 1

### Purpose

The customer dimension consolidates customer attributes from the Silver layer and provides a stable lookup key for downstream order analysis.

### Processing workflow

1. Read the unified customer records from `silver.customers_unified`.
2. Filter records using the selected `file_run_date`.
3. Derive `customer_sk` by applying SHA-256 hashing to `mobile_number`.
4. Use `ROW_NUMBER()` partitioned by mobile number and ordered by ingestion timestamp to select the latest record within the batch.
5. Use a Delta Lake `MERGE` to match existing customer records by mobile number.
6. Update existing customer attributes when a match is found; insert a new record when no match exists.
7. Maintain `last_updated_ts` to record when a customer dimension record is updated.

### SCD Type 1 behaviour

When an existing customer is encountered again, the dimension is updated with the latest attributes. Previous attribute values are not maintained as historical versions.

### Target table

`gold.customers`

### Outcome

A consolidated customer dimension that supports customer lookups from both online and offline order records.

In [0]:
%sql
create table if not exists `spark-databricks-project-catalog`.gold.customers --if table exists in rerun case, this    will be skipped
using delta
as 
select sha2(mobile_number, 256) as customer_sk, --surrogate key (sk), keep it in fact table too
       full_name,
       email,
       mobile_number,
       city,
       state,
       zipcode,
       signup_date,
       offline_customer_id,
       online_customer_id,
       file_date,
       source,
       ingest_ts,
       current_timestamp() as last_updated_ts --new column as we need SCD type 1 data .ie. only latest
 from `spark-databricks-project-catalog`.silver.customers_unified
 where 1 = 2;  

num_affected_rows,num_inserted_rows


In [0]:
%sql
----SCD Type 1

merge into `spark-databricks-project-catalog`.gold.customers t
using (
    select * from (
        select sha2(mobile_number, 256) as customer_sk,
           full_name,
           email,
           mobile_number,
           city,
           state,
           zipcode,
           signup_date,
           offline_customer_id,
           online_customer_id,
           file_date,
           source,
           ingest_ts,
           row_number() over (partition by mobile_number order by ingest_ts desc) as rn
         from `spark-databricks-project-catalog`.silver.customers_unified
         where file_date = :file_run_date
    )
    where rn = 1
) s
on t.mobile_number = s.mobile_number

--if same mobile no.(customer) is appearing again
when matched then update set  
  t.customer_sk = s.customer_sk,
  t.full_name = s.full_name,
  t.email = s.email,
  t.city = s.city,
  t.state = s.state,
  t.zipcode = s.zipcode,
  t.signup_date = s.signup_date,
  t.offline_customer_id = s.offline_customer_id,
  t.online_customer_id = s.online_customer_id,
  t.file_date = s.file_date,
  t.source = s.source,
  t.ingest_ts = s.ingest_ts,
  t.last_updated_ts = current_timestamp()

when not matched then insert (
  customer_sk,
  full_name,
  email,
  mobile_number,
  city,
  state,
  zipcode,
  signup_date,
  offline_customer_id,
  online_customer_id,
  file_date,
  source,
  ingest_ts,
  last_updated_ts
)
values (
   s.customer_sk,
   s.full_name,
   s.email,
   s.mobile_number,
   s.city,
   s.state,
   s.zipcode,
   s.signup_date,
   s.offline_customer_id,
   s.online_customer_id,
   s.file_date,
   s.source,
   s.ingest_ts,
   current_timestamp()
);


num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
200103,200103,0,0


## 2. Product Dimension — SCD Type 2

### Purpose

Product attributes such as name, category, brand, price, tax rate, currency, and active status may change over time. The product dimension preserves these changes so that orders can be associated with the appropriate product version.

### Processing workflow

1. Read product records from `silver.products_unified` for the selected `file_run_date`.
2. Create a temporary view named `v_products_run`.
3. Generate `product_sk` using SHA-256 hashing of `sku_id` and `file_date`.
4. Calculate a change hash from the selected product attributes to identify changes between product versions.
5. Use `ROW_NUMBER()` to select the latest ingested record for each SKU in the current batch.
6. Use a Delta Lake `MERGE` to expire the existing current record when a change is detected.
7. Set the expired record's `effective_to` date to the day before the new version's file date.
8. Insert new SKUs and changed product versions with updated attributes and effective dates.

### SCD Type 2 fields

| Field | Purpose |
|---|---|
| `product_sk` | Identifies a particular product version |
| `sku_id` | Business identifier used to match product records |
| `effective_from` | Date from which the product version is valid |
| `effective_to` | End date of the product version's validity |
| `is_current` | Indicates whether the record is the current product version |

### Historical lookup

The order-processing logic joins products using `sku_id` and checks whether the order's file date falls between `effective_from` and `effective_to`. This enables the fact table to reference the product version valid for that date.

### Target table

`gold.products`

### Outcome

A product dimension that retains historical versions for changed products while distinguishing the current version from expired records.

In [0]:
%sql
create table if not exists `spark-databricks-project-catalog`.gold.products
( 
    product_sk string,
    sku_id string,
    product_name string,
    category string,
    brand string,
    price float,
    currency string,
    is_active string,
    tax_rate float,
    file_date date,
    source string,
    ingest_ts timestamp,
    effective_from date, -- SCD Type 2
    effective_to date,  -- SCD Type 2
    is_current boolean --- if current record then 1 else 0
)
using delta;  ---not requried as by default it creates a Delta table

In [0]:
%sql

-- sku_id , price , file_run_date , efeective_from , expire_date
-- sku1 , 100 , 20260101 , 1-jan, 10-jan
-- sku1 , 110 , 20260111 , 11jan , 9999-12-31

-- new load/change happening on another day -> sku1 , 110 , 20260111 - need to expire the old/existing record and maintain new record with latest info
-- sku2 , 110 , 20260111 - here only add as new sku_id has appeared

-- creating a view so that the change once captured (change_hash), could be use while update & insert
CREATE OR REPLACE TEMP VIEW v_products_run AS
SELECT
    sku_id,
    product_name,
    category,
    brand,
    price,
    tax_rate,
    currency,
    is_active,
    file_date,
    source,
    ingest_ts,

    sha2(
        concat_ws('||', sku_id, file_date),
        256
    ) AS product_sk, -- surrogate key based on sku_id & date, as sku_id can repeat while maintaining history, hence to keep it unique surrogate key is a combination of sku_id & file_date

    sha2(
        concat_ws(
            '||',
            product_name,
            category,
            brand,
            price,
            tax_rate,
            currency,
            is_active
        ),
        256
    ) AS change_hash -- to identify if there is any change in the product details

FROM `spark-databricks-project-catalog`.silver.products_unified

WHERE file_date = :file_run_date; -- for current date, created a temp view

In [0]:
%sql
DESCRIBE v_products_run;

col_name,data_type,comment
sku_id,string,null
product_name,string,null
category,string,null
brand,string,null
price,float,null
tax_rate,float,null
currency,string,null
is_active,string,null
file_date,string,null
source,string,null


In [0]:
%sql

-- Implementing SCD Type 2


merge into `spark-databricks-project-catalog`.gold.products t
using (
    select * from (
        select *,
            row_number() over (partition by sku_id order by ingest_ts desc) as rn
        from v_products_run
    )
    where rn = 1  --only taking latest sku_id record 
) s
on t.sku_id = s.sku_id and 
   t.is_current = True -- whatever records r coming again from source, on the basis of sku_id & is_current = True, and if there is a change (catching by below hash ) then we will expire (update) the old record

when matched and 
   sha2(concat_ws('||', 
                     t.product_name,
                     t.category,
                     t.brand,
                     t.price,
                     t.currency,
                     t.is_active,
                     t.tax_rate), 256) <> s.change_hash --again we're craeting a change hash in target & if there is a change then we will update the target details as below
then update set
    t.is_current = False,
    t.effective_to = date_sub( to_date(:file_run_date, 'yyyyMMdd'), 1)

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
4998,4998,0,0


In [0]:
%sql

-- Update done above, now Insert whatever is new and update too

INSERT INTO `spark-databricks-project-catalog`.gold.products
SELECT
    s.product_sk,
    s.sku_id,
    s.product_name,
    s.category,
    s.brand,
    s.price,
    s.currency,
    s.is_active,
    s.tax_rate,
    to_date(s.file_date, 'yyyyMMdd') AS file_date,
    s.source,
    s.ingest_ts,
    to_date(CAST(s.file_date AS STRING), 'yyyyMMdd') AS effective_from,
    TIMESTAMP '9999-12-31' AS effective_to,
    TRUE AS is_current
FROM v_products_run s --has data from silver.products_unified

LEFT JOIN `spark-databricks-project-catalog`.gold.products t
    ON s.sku_id = t.sku_id
    AND t.is_current = TRUE

WHERE t.sku_id IS NULL
   OR sha2(concat_ws(
                   '||',
                   t.product_name,
                   t.category,
                   t.brand,
                   t.price,
                   t.currency,
                   t.is_active,
                   t.tax_rate
                 ), 256) <> s.change_hash;

num_affected_rows,num_inserted_rows
9996,9996


## 3. Order Fact Table — Dimension Lookups

### Purpose

The order fact table brings together unified order measures and attributes with the customer and product surrogate keys required for dimensional analysis.

### Processing workflow

1. Read orders from `silver.orders_unified` for the selected `file_run_date`.
2. Match each order's customer ID against the online and offline customer identifiers in `gold.customers`.
3. Match each order's SKU against `gold.products`.
4. Select the product version whose effective date range includes the order's file date.
5. Insert successfully matched records into `gold.fact_orders`.
6. Route records with missing customer or product lookup results to `silver.reject_orders`.

### Fact table attributes

The table includes order ID, order timestamp, customer and product surrogate keys, quantity, list price, discount, payment mode, store number, file date, source system, and ingestion timestamp.

### Data quality and rejects

Orders that cannot be matched to a customer or product dimension record are captured in `silver.reject_orders` with a `reject_reason` indicating whether the customer or product was not found.

This provides visibility into lookup failures instead of silently excluding those records from the fact table.

### Target table

`gold.fact_orders`

### Outcome

An analytics-ready order fact table that connects retail transactions to customer and product dimensions, with a separate table for unresolved dimension lookups.

In [0]:
%sql
create table if not exists `spark-databricks-project-catalog`.gold.fact_orders 
(
    order_id string,
    order_ts timestamp,

    customer_sk string,
    product_sk string,
    
    qty int,
    list_price double,
    discount double,

    payment_mode string,
    store_num string,

    file_date string,
    source_system string,
    ingest_ts timestamp
)
 using delta;

## 4. Data Quality — Reject Handling

### Purpose

Fact-table loading requires valid customer and product dimension lookups. Orders with unresolved dimension references are identified separately for investigation.

### Reject workflow

- Match order customer IDs against the online and offline customer IDs in the Gold customer dimension.
- Match order SKUs against the Gold product dimension using the product's effective-date range.
- Identify orders for which either lookup fails.
- Assign a `reject_reason` of `customer not found` or `product not found`.
- Store the rejected records in `silver.reject_orders` for review and subsequent remediation.

### Why this matters

Separating rejected records makes lookup failures visible and helps prevent unresolved dimension references from entering the analytics-ready fact table.

In [0]:
%sql
--- reject orders with no lookup data
create table if not exists `spark-databricks-project-catalog`.silver.reject_orders
as (
    select *,
        cast('dummy' as varchar(100)) as reject_reason 
     from `spark-databricks-project-catalog`.silver.orders_unified
     where 1 = 2
    );

delete from `spark-databricks-project-catalog`.silver.reject_orders
 where file_date = :file_run_date;
 --using (:) -> as its a variable

insert into `spark-databricks-project-catalog`.silver.reject_orders -- rejected records would be inserted into this

select o.*,
       case when c.customer_sk is null then 'customer not found'
            else 'product not found'
            end 
            as reject_reason
 from `spark-databricks-project-catalog`.silver.orders_unified o
 
 -- ✅ customer lookup
 left join `spark-databricks-project-catalog`.gold.customers c
 on o.customer_id = c.offline_customer_id -- Take the customer ID from the order and check whether    that                                         customer exists in the Gold Customer table.
 or o.customer_id = c.online_customer_id

-- ✅ product SCD2 lookup
 left join `spark-databricks-project-catalog`.gold.products p
  on o.sku_id = p.sku_id 
  and to_date(o.file_date, 'yyyyMMdd') between p.effective_from and p.effective_to
where o.file_date = :file_run_date and 
(c.customer_sk is null OR p.product_sk is null);

--Take the SKU from the order and find the product version that was valid on that order's date.

num_affected_rows,num_inserted_rows
4314140,4314140


In [0]:
%sql
delete from `spark-databricks-project-catalog`.gold.fact_orders where 
             file_date = :file_run_date;
insert into `spark-databricks-project-catalog`.gold.fact_orders
select 
       o.order_id,
       o.order_ts,
       c.customer_sk,
       p.product_sk,
       o.qty,
       o.list_price,
       o.discount,
       o.payment_mode,
       o.store_num,
       o.file_date,
       o.source_system,
        o.ingestion_time
 from `spark-databricks-project-catalog`.silver.orders_unified o 

-- ✅ customer lookup
 inner join `spark-databricks-project-catalog`.gold.customers c 
 on o.customer_id = c.online_customer_id
 or o.customer_id = c.offline_customer_id

-- ✅ product SCD2 lookup
 inner join `spark-databricks-project-catalog`.gold.products p 
 on o.sku_id = p.sku_id
 and to_date(o.file_date, 'yyyyMMdd') between p.effective_from and p.effective_to
 where o.file_date = :file_run_date;      
 --not only always latest data, but whenever catching the data, the record should've been active
 -- based on file date, we've to get the effective surrogate key

num_affected_rows,num_inserted_rows
29274460,29274460


## 5. Gold Layer — Output Summary

### Target tables

| Table | Role |
|---|---|
| `gold.customers` | Customer dimension using SCD Type 1 |
| `gold.products` | Product dimension using SCD Type 2 |
| `gold.fact_orders` | Order fact table enriched with dimension surrogate keys |
| `silver.reject_orders` | Stores orders with unresolved customer or product lookups |

### Key Data Engineering Concepts Demonstrated

- Dimensional modelling using fact and dimension tables.
- SCD Type 1 customer dimension management.
- SCD Type 2 product history management.
- Surrogate-key generation using SHA-256 hashing.
- Change detection using attribute hashes.
- Delta Lake `MERGE` operations.
- Window functions for selecting the latest source record.
- Effective-date joins for historical product lookup.
- Data quality checks and reject handling.
- Date-parameterized batch processing.

### End-to-End Architecture

**Bronze → Silver → Gold**

- **Bronze:** Ingests source datasets and preserves ingestion metadata.
- **Silver:** Standardizes and unifies customer, product, and order records.
- **Gold:** Builds dimensional tables and a fact table for downstream analysis.

The three layers form a complete lakehouse workflow from raw ingestion to analytics-ready data.